# 02 — Stationarity Analysis

## Project: Improving Basket Trading Using Bayesian Optimization

### Objective
This notebook investigates the time-series properties of five U.S. sector ETFs to assess whether their price and return series exhibit evidence of stationarity.

Stationarity analysis is an important preliminary step before investigating long-run relationships and cointegration.

### Research Questions
1. Do ETF prices exhibit evidence of unit roots?
2. Are daily ETF returns more consistent with stationary behavior?
3. How do ADF and KPSS test results compare?
4. What are the implications for subsequent cointegration analysis?

### Assets
XLK, XLF, XLE, XLI, XLP

### Data Period
January 2015 – October 2026

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "etf_prices.csv"

# Load ETF prices
prices = pd.read_csv(
    DATA_PATH,
    index_col="Date",
    parse_dates=True
)

# Calculate daily returns
returns = prices.pct_change().dropna()

# Verify datasets
print("Prices shape:", prices.shape)
print("Returns shape:", returns.shape)

print("\nPrice date range:")
print(prices.index.min(), "to", prices.index.max())

Prices shape: (2972, 5)
Returns shape: (2971, 5)

Price date range:
2015-01-02 00:00:00 to 2026-10-27 00:00:00


## 2. Augmented Dickey-Fuller (ADF) Test
The Augmented Dickey-Fuller test is used to investigate the presence of a unit root in a time series.

### Hypotheses
- **H0:** The series contains a unit root.
- **H1:** The series is stationary around a constant under the selected model specification.

We use a significance level of 5%.

We first compare XLK price levels and daily returns to investigate how their statistical properties differ.

The ADF test is implemented with an intercept and automatic lag selection based on the Akaike Information Criterion (AIC).

In [3]:
from statsmodels.tsa.stattools import adfuller

# ADF test on XLK prices
adf_prices = adfuller(
    prices["XLK"],
    regression="c",
    autolag="AIC"
)

# ADF test on XLK daily returns
adf_returns = adfuller(
    returns["XLK"],
    regression="c",
    autolag="AIC"
)

print("ADF TEST - XLK PRICES")
print("ADF Statistic:", adf_prices[0])
print("p-value:", adf_prices[1])
print("Lags used:", adf_prices[2])

print("\nADF TEST — XLK DAILY RETURNS")
print("ADF Statistic:", adf_returns[0])
print("p-value:", adf_returns[1])
print("Lags used:", adf_returns[2])

ADF TEST — XLK PRICES
ADF Statistic: 1.6933703770972843
p-value: 0.9981123290113258
Lags used: 29

ADF TEST — XLK DAILY RETURNS
ADF Statistic: -18.492158035155043
p-value: 2.1260849237220475e-30
Lags used: 8


/var/folders/c7/tww1vnb53ssdxbfssfglvfb40000gp/T/ipykernel_30940/3155055260.py:4: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_prices = adfuller(
/var/folders/c7/tww1vnb53ssdxbfssfglvfb40000gp/T/ipykernel_30940/3155055260.py:11: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_returns = adfuller(


In [4]:
# Verify ADF results for XLK

print("XLK PRICES")
print(f"ADF Statistic: {adf_prices[0]:.6f}")
print(f"p-value: {adf_prices[1]:.12g}")
print(f"Lags used: {adf_prices[2]}")

print("\nXLK DAILY RETURNS")
print(f"ADF Statistic: {adf_returns[0]:.6f}")
print(f"p-value: {adf_returns[1]:.12g}")
print(f"Lags used: {adf_returns[2]}")

print("\nHYPOTHESIS DECISIONS")

alpha = 0.05

for name, result in [
    ("XLK Prices", adf_prices),
    ("XLK Returns", adf_returns)
]:
    if result[1] < alpha:
        print(f"{name}: Reject H0")
    else:
        print(f"{name}: Fail to reject H0")

XLK PRICES
ADF Statistic: 1.693370
p-value: 0.998112329011
Lags used: 29

XLK DAILY RETURNS
ADF Statistic: -18.492158
p-value: 2.12608492372e-30
Lags used: 8

HYPOTHESIS DECISIONS
XLK Prices: Fail to reject H0
XLK Returns: Reject H0


## 3. ADF Tests Across All ETFs

This section extends the Augmented Dickey-Fuller analysis to all five sector ETFs.

We test both price levels and daily returns using an intercept and automatic lag selection based on AIC.

The null hypothesis is that the series contains a unit root.

The significance level is set to 5%.

Results are organized into a comparative table to identify differences between price and return series.

In [8]:
# ADF tests for all ETFs

alpha = 0.05
adf_results = []

for etf in prices.columns:
    for series_type, dataset in [
        ("Prices", prices),
        ("Returns", returns)
    ]:
        result = adfuller(
        dataset[etf],
        regression="c",
        autolag="AIC",
        result_object=False
    )

        adf_results.append({
            "ETF": etf,
            "Series": series_type,
            "ADF Statistic": result[0],
            "p-value": result[1],
            "Lags": result[2],
            "Decision": (
                "Reject H0"
                if result[1] < alpha
                else "Fail to reject H0"
            )
        })

adf_df = pd.DataFrame(adf_results)

display(adf_df.round(6))

,ETF,Series,ADF Statistic,p-value,Lags,Decision
0,XLK,Prices,1.693370,0.998112,29,Fail to reject H0
1,XLK,Returns,-18.492158,0.000000,8,Reject H0
2,XLF,Prices,0.297539,0.977231,27,Fail to reject H0
3,XLF,Returns,-17.347767,0.000000,8,Reject H0
4,XLE,Prices,1.107105,0.995277,0,Fail to reject H0
5,XLE,Returns,-10.699355,0.000000,28,Reject H0
6,XLI,Prices,0.306534,0.977642,9,Fail to reject H0
7,XLI,Returns,-17.228909,0.000000,8,Reject H0
8,XLP,Prices,-0.065114,0.952804,27,Fail to reject H0
9,XLP,Returns,-11.826921,0.000000,26,Reject H0
